In [1]:
%pip install gym torch -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
env = gym.make("CartPole-v1")

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

In [4]:
class DQN(nn.Module):
    def __init__(self):
        super(DQN, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(state_size, 24),
            nn.ReLU(),
            nn.Linear(24, 24),
            nn.ReLU(),
            nn.Linear(24, action_size)
        )

    def forward(self, x):
        return self.fc(x)

model = DQN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

In [5]:
memory = deque(maxlen=2000)

gamma = 0.95
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01
batch_size = 32

In [6]:
def choose_action(state):

    if random.random() < epsilon:
        return random.randint(0, action_size - 1)

    state = torch.FloatTensor(state)
    q_values = model(state)

    return torch.argmax(q_values).item()

In [7]:
def train():

    if len(memory) < batch_size:
        return

    batch = random.sample(memory, batch_size)

    for state, action, reward, next_state, done in batch:

        state = torch.FloatTensor(state)
        next_state = torch.FloatTensor(next_state)

        target = reward

        if not done:
            target += gamma * torch.max(model(next_state)).item()

        target_f = model(state).detach().clone()
        target_f[action] = target

        output = model(state)

        loss = loss_fn(output, target_f)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [8]:
episodes = 50

for episode in range(episodes):

    state, _ = env.reset()
    total_reward = 0

    done = False

    while not done:

        action = choose_action(state)

        next_state, reward, terminated, truncated, _ = env.step(action)

        done = terminated or truncated

        memory.append((state, action, reward, next_state, done))

        state = next_state
        total_reward += reward

        train()

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    print(f"Episode: {episode+1}, Score: {total_reward}")

c:\Users\Personal\anaconda3\Lib\site-packages\gym\utils\passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Episode: 1, Score: 16.0
Episode: 2, Score: 21.0
Episode: 3, Score: 11.0
Episode: 4, Score: 13.0
Episode: 5, Score: 15.0
Episode: 6, Score: 16.0
Episode: 7, Score: 16.0
Episode: 8, Score: 20.0
Episode: 9, Score: 12.0
Episode: 10, Score: 18.0
Episode: 11, Score: 62.0
Episode: 12, Score: 13.0
Episode: 13, Score: 19.0
Episode: 14, Score: 21.0
Episode: 15, Score: 27.0
Episode: 16, Score: 18.0
Episode: 17, Score: 20.0
Episode: 18, Score: 35.0
Episode: 19, Score: 13.0
Episode: 20, Score: 11.0
Episode: 21, Score: 17.0
Episode: 22, Score: 11.0
Episode: 23, Score: 9.0
Episode: 24, Score: 14.0
Episode: 25, Score: 23.0
Episode: 26, Score: 16.0
Episode: 27, Score: 18.0
Episode: 28, Score: 13.0
Episode: 29, Score: 16.0
Episode: 30, Score: 12.0
Episode: 31, Score: 37.0
Episode: 32, Score: 20.0
Episode: 33, Score: 20.0
Episode: 34, Score: 48.0
Episode: 35, Score: 20.0
Episode: 36, Score: 21.0
Episode: 37, Score: 39.0
Episode: 38, Score: 40.0
Episode: 39, Score: 24.0
Episode: 40, Score: 14.0
Episode: 4

In [9]:
state, _ = env.reset()

done = False

while not done:

    state = torch.FloatTensor(state)

    action = torch.argmax(model(state)).item()

    next_state, reward, terminated, truncated, _ = env.step(action)

    done = terminated or truncated

    state = next_state

env.close()